# OSRT v6 — GRPO on Colab (G4 / RTX PRO 6000 Blackwell, 96GB)

RL with verifiable maths rewards, starting from the SFT-v4 checkpoint soup (**20.0% GSM8K reasoning-on vs 13.5% off**, n=200, measured 2026-08-10).

**Runtime: G4.** 96GB — more than an H100 — and sm_120 is verified end-to-end for this model. midtrain3 measured ~6,300 tok/s compiled, about 63% of H100 (this model is compute-bound, so GDDR7's lower bandwidth barely bites).

**Secrets:** add `HF_TOKEN` and `WANDB_API_KEY` in the Colab key sidebar (🔑).

**Why this exists rather than the Modal stage:** the Modal GRPO loop generates one prompt at a time (batch 16) and computes log-probs one sequence at a time (batch 1, twice per rollout). Measured **~5.5 min/step** — 900 steps would be 75 hours. `osrt.grpo_train` batches both. **Measured ~90s/step** on G4 at 16 prompts × 16 rollouts, so 900 steps ≈ 22 hours across several sessions (~100 steps per 2.6h session before the VM is reclaimed).

---
### Read this before launching: wave 1 made the model WORSE
100 steps took held-out GSM8K from **20.0% → 12.5%**, monotonically, in both reasoning modes and under both personas. Root cause was the reward, not the loop: `correctness_partial_credit` paid `within_5_pct` **+3.5** against exact's **+5.0**, so a wrong answer earned 70% of a correct one, and group-normalised advantage promoted the best near-miss to a large positive advantage. Fixed in `9225297` — only an exact answer scores positive now.

**What fooled us:** truncation collapsed 31/256 → 2/256 and reward crossed positive. Both were *format* acquisition. Reward going up is not evidence of learning — run cell 6.

---
### The four Colab fixes (each cost a debugging round on midtrain3 — do not skip)
1. **`--auth=adc`** on every `colab` CLI call. oauth2 silently drops the `colaboratory` scope on refresh → keep-alive 403s → VM reclaimed mid-run.
2. **No DataLoader workers.** Spawned workers hit a fatal `PyGILState_Release` teardown race (tokenizers/pyarrow + torch). This script uses none.
3. **`PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True`** — set by the script itself.
4. **Read progress from W&B / HF, not `colab exec`** — its websocket reports a false "step 0" when the run is much further along.

In [ ]:
# ── 1. Repo + deps + secrets ─────────────────────────────────────────
import os, sys

BRANCH = "feat/sft-harvest"
if not os.path.isdir("/content/osrt"):
    !git clone -q --depth 1 -b {BRANCH} https://github.com/CodeHalwell/OSRT-605M-A269M.git /content/osrt
%pip -q install -U "transformers>=5.3.0" datasets tokenizers safetensors wandb huggingface_hub
sys.path.insert(0, "/content/osrt/src")

from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
cap = torch.cuda.get_device_capability(0)
print(torch.cuda.get_device_name(0), f"| sm_{cap[0]}{cap[1]} |",
      f"{torch.cuda.get_device_properties(0).total_memory/2**30:.0f}GB | torch", torch.__version__)

In [ ]:
# ── 2. Prompt set: UNSEEN problems with numeric gold ─────────────────
# Must be problems SFT-v4 never trained on. Its builder consumed the head of
# the orca-math stream (6,000 kept), so we skip well past that. Using
# GSM8K-train instead would be RL on memorised solutions — healthy reward, no
# generalisation, and a failure that looks like the model simply plateauing.
import json, os
from datasets import load_dataset
sys.path.insert(0, "/content/osrt/src")
from osrt.rewards import extract_numeric_answer

OUT, TARGET, SKIP = "/content/grpo_prompts.jsonl", 6000, 60_000
if not os.path.exists(OUT):
    ds = load_dataset("microsoft/orca-math-word-problems-200k",
                      split="train", streaming=True).skip(SKIP)
    kept = []
    for row in ds:
        if len(kept) >= TARGET:
            break
        q = (row.get("question") or "").strip()
        gold = extract_numeric_answer(row.get("answer") or "")
        if q and gold is not None:
            kept.append({"question": q, "answer": str(gold).strip()})
    with open(OUT, "w") as f:
        for r in kept:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"wrote {len(kept)} prompts")
print(OUT, sum(1 for _ in open(OUT)), "prompts")

# NOTE ON DIFFICULTY SCREENING: ~62-73% of unfiltered prompts are "dead" —
# all rollouts fail, so under group-normalised advantage they contribute
# EXACTLY zero gradient. Screening them out makes each wave ~2.6x more useful,
# but measured at ~13 scans/min it costs hours. Dead prompts are wasteful, not
# harmful, and generation is only a fraction of a step — so we train unfiltered
# and spend the time on more STEPS instead. Revisit if steps get cheap.

In [ ]:
# ── 3. Sanity: 3 steps, tiny wave. Run this BEFORE the long launch. ──
# Checks what no config review can: that the batched loop actually trains,
# VRAM fits, and — most importantly — that the rollouts look sane. The single
# most valuable thing here is READING THE GENERATIONS: a missing system prompt
# once made the model skip <|think|> entirely and eat a -0.5 ambiguity penalty
# on every rollout, which no memory-or-timing check would have caught.
!cd /content/osrt && python scripts/colab_grpo.py \
    --ckpt-dir /content/ckpt --prompts /content/grpo_prompts.jsonl \
    --tokenizer /content/osrt/v6_tokenizer_export \
    --hf-repo HallD/osrt-v6-ckpt \
    --total-steps 3 --num-prompts 4 --ckpt-interval 999 --no-wandb

### Read the sanity output before going further

**Rollouts must look like** `<|think|>…working…<|/think|><|answer|>72<|/answer|>` — think block present, a **single bare number** in the answer block, clean stop.

**Red flags:**
- Completions starting `<|answer|>` with no think block → the system prompt isn't reaching the model.
- Several numbers inside `<|answer|>` → strict extraction rules them `ambiguous` and applies −0.5; reward will sit negative.
- `live 0/N` → every rollout in every group scored identically, so all advantages are zero and **nothing is learning**. That is the recorded collapse mode ("uniform rewards → zero advantage → frozen updates"). Raise `group_size` or check the reward path.
- Reward strongly negative at step 0 → something is wrong; step 0 should be mildly positive since format alone is worth up to +3.0.

Also note **seconds/step** and **VRAM** — they set the wave size and total steps below.

In [ ]:
# ── 4. THE RUN — foreground, so the session stays alive ──────────────
# FOREGROUND, not nohup. A detached launch returns instantly, the notebook
# then looks IDLE, and Colab reclaims the VM out from under the background
# process — the run dies for the very reason the nohup was meant to prevent.
# A foreground cell streaming output keeps the session active (the script
# prints with flush=True, so `tee` streams live rather than buffering).
#
# Losing the cell is survivable: checkpoints push to HF every 10 steps, so a
# drop costs at most 10 steps (~15 min). Re-run this cell to resume — the
# script scans --ckpt-dir for the newest grpo_v6_step_*.pt, then falls back to
# HF, then to the base checkpoint.

# On a FRESH VM /content/ckpt is empty, so the run starts from the SFT-v4 soup.
# Set this to carry on from a pushed GRPO checkpoint instead.
RESUME_FROM = "grpo_v6_step_100.pt"   # "" = fresh start from the soup

import os, shutil
os.makedirs("/content/ckpt", exist_ok=True)
if RESUME_FROM and not os.path.exists(f"/content/ckpt/{RESUME_FROM}"):
    from huggingface_hub import hf_hub_download
    src = hf_hub_download("HallD/osrt-v6-ckpt", RESUME_FROM, repo_type="model")
    shutil.copy2(src, f"/content/ckpt/{RESUME_FROM}")
    print(f"staged {RESUME_FROM} -> /content/ckpt (will resume from it)")
elif not RESUME_FROM:
    print("fresh start: no local step ckpt, so the SFT-v4 soup is the base")

!cd /content/osrt && PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
    python scripts/colab_grpo.py \
    --ckpt-dir /content/ckpt \
    --prompts /content/grpo_prompts.jsonl \
    --tokenizer /content/osrt/v6_tokenizer_export \
    --hf-repo HallD/osrt-v6-ckpt \
    --ckpt-interval 10 \
    --num-prompts 16 \
    --micro-batch 24 \
    --max-gen-len 768 \
    --compile 2>&1 | tee -a /content/grpo.log

In [ ]:
# ── 6. Held-out GSM8K eval of pushed checkpoints ─────────────────────
# WHY: the per-step `acc` in the log is dominated by WHICH 16 prompts were
# drawn, not by model change — it swung 5.9/10.5/9.4/16.0/9.4/4.7/21.9% over
# the first 60 steps. Only a FIXED problem set separates learning from draw
# noise. Same eval and same n as the SFT sweep, so numbers are comparable.
#
# BASELINES, measured 2026-08-10 at n=200 (these SUPERSEDE the old 17.5/12.0,
# which did not reproduce on a re-run of the same soup file):
#     SFT-v4 soup @ instruction_strict : acc_on 20.0%  acc_off 13.5%  +6.5pp
#     SFT-v4 soup @ minimal_format     : acc_on 17.0%  acc_off 14.0%  +3.0pp
#
# Eval noise floor is ~0.5pp at n=200 (measured: identical weights and persona
# across two separate processes moved by one problem). Greedy decode with fixed
# personas and the first 200 GSM8K test rows, so it is near-reproducible.
# Do NOT drop to n=50 — that cannot resolve the moves we care about.
#
# SCORE BOTH PERSONAS. GRPO trains under `minimal_format`; the eval's default
# is `instruction_strict`. Wave 1's damage showed up as the two CONVERGING —
# a healthy soup differs by 3pp between them, while both damaged checkpoints
# scored identically under both, i.e. the model stopped conditioning on the
# system prompt at all. Trim to [""] to halve the runtime.
CKPTS = ["grpo_v6_step_110.pt", "grpo_v6_step_150.pt", "grpo_v6_step_200.pt"]
PERSONAS = ["", "minimal_format"]      # "" = default (instruction_strict)
N_PROBLEMS = 200

import torch, sys
sys.path.insert(0, "/content/osrt/src")
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer
from osrt.model import OSRTForCausalLM
from osrt.presets import build_config
from osrt.sft_eval import run_reasoning_eval

tok = AutoTokenizer.from_pretrained("/content/osrt/v6_tokenizer_export")
cfg = build_config(vocab_size=len(tok), real_vocab_size=len(tok),
                   bos_token_id=tok.bos_token_id, eos_token_id=tok.eos_token_id,
                   pad_token_id=tok.pad_token_id, fused_cross_entropy_chunks=8)
device = torch.device("cuda")
model = OSRTForCausalLM(cfg).to(device)

rows = []
for name in CKPTS:
    try:
        p = hf_hub_download("HallD/osrt-v6-ckpt", name, repo_type="model")
    except Exception as e:
        print(f"{name}: not on HF yet ({type(e).__name__})"); continue
    ck = torch.load(p, map_location=device, weights_only=True)
    missing, unexpected = model.load_state_dict(
        ck.get("model_state_dict", ck), strict=False)
    assert not missing and not unexpected, f"{name}: {missing[:3]} {unexpected[:3]}"
    del ck
    model.eval()
    # telemetry off: its .item() calls are pure CUDA syncs at inference
    if hasattr(model, "set_moe_telemetry"):
        model.set_moe_telemetry(False)
    for persona in PERSONAS:
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            m = run_reasoning_eval(model, tok, device, n_problems=N_PROBLEMS,
                                   max_new_tokens=512, batch_size=32,
                                   repetition_penalty=1.2,
                                   on_persona=persona)
        rows.append((name, persona, m))
        print(f"{name:<24} @{m['sft_eval/persona_on']:<20} "
              f"acc_on {100*m['sft_eval/acc_on']:5.1f}%  "
              f"acc_off {100*m['sft_eval/acc_off']:5.1f}%  "
              f"delta {100*m['sft_eval/acc_delta_on_minus_off']:+5.1f}pp  "
              f"fmt {100*m['sft_eval/format_ok_on']:5.1f}%  "
              f"len {m['sft_eval/resp_len_on']:.0f}", flush=True)

print("\nBASELINE (SFT-v4 soup, n=200):")
print("  @instruction_strict  acc_on 20.0%  acc_off 13.5%  +6.5pp")
print("  @minimal_format      acc_on 17.0%  acc_off 14.0%  +3.0pp")
if rows:
    best = max(rows, key=lambda r: r[2]['sft_eval/acc_on'])
    print(f"best so far: {best[0]} @{best[1] or 'default'} "
          f"at {100*best[2]['sft_eval/acc_on']:.1f}%")
    # Prompt-insensitivity check: the two personas converging is the wave-1
    # damage signature, independent of the absolute accuracy.
    for name in dict.fromkeys(r[0] for r in rows):
        per = {r[1]: r[2]['sft_eval/acc_on'] for r in rows if r[0] == name}
        if len(per) == 2:
            spread = 100 * abs(per[""] - per["minimal_format"])
            flag = "  <-- personas converged (soup spread is 3.0pp)" if spread < 1.0 else ""
            print(f"  {name}: persona spread {spread:.1f}pp{flag}")

In [ ]:
# ── 5. Log digest (the run streams inline above; this greps the tee'd log) ──
# Useful for skimming a long run, or after a reconnect when the cell output
# is gone but /content/grpo.log survives. Note `tee -a` APPENDS, so this
# spans every resumed session on this VM.
!grep -E "^step|rollouts @|^  \[|saved|pushed|Error|Traceback|CUDA out of memory" \
    /content/grpo.log | tail -40

## Judging the run

**Reward EMA is not the judge, and wave 1 proved it.** Reward crossed positive and truncation collapsed 31/256 → 2/256 while held-out GSM8K fell **20.0% → 12.5%**. The format term alone is worth up to +3.0 and the model already scores ~99% on format, so an early rise is *format consolidation*. Loss and accuracy have now dissociated four separate times on this project.

**The only judge is cell 6.** Run it at ~step 50 and ~150 — not at 300. It is cheap next to the credits a bad trajectory burns.

**What to watch in the step line:**
- **`acc`** — useful only as a floor check. On 16 prompts it swung 4.7–21.9% over 60 steps, so a single step's value means nothing.
- **`live N/M`** — rollouts with non-zero advantage. Collapsing toward 0 means learning has stopped whatever reward does. With the tiers removed, watch this early: the proximity credit was partly propping it up, and the graded negative tiers (−0.5 / −2.0 / −2.5) are now what supplies the variance.
- **`kl`** — a jump well off trend (wave 1 went 0.048 → 0.33 in ten steps) means the policy is drifting off the frozen reference. `kl_coeff` 0.15 did not hold it.
- **The printed rollouts** — is the reasoning working the problem, or reciting its shape? Wave 1 kept perfect format while inventing numbers.

**Two failure signatures now on record:**
1. **Prompt-insensitivity collapse** — the two personas in cell 6 converging. A healthy soup differs by 3.0pp; both damaged checkpoints scored identically under both, meaning the model had stopped conditioning on the system prompt at all.
2. **Format decay with length inflation** — `fmt_on` 99 → 96% while `len_on` grew 388 → 430. Format getting *worse* late while lengths grow is degeneration, not exploration.

**Sweep checkpoints at the end, don't ship the last one.** In SFT-v4 the final checkpoint was measurably *not* the best: accuracy peaked at step 1,800 and the reasoning-on advantage eroded from +7.0pp to +1.0pp as training extended. Wave 1's best checkpoint was step 0.